In [1]:
# Tecnología
import json
import calendar
import pandas as pd
from sparky_bc import Sparky
import datetime as dt
from dateutil.relativedelta import relativedelta

# files lz conection
path_sparky_conf = '/Users/santlond/Documents/sparky_conf.json'

# Configurar conexión a LZ
with open(path_sparky_conf, 'rb') as JSON_lz_File:
    sp_config = json.loads(JSON_lz_File.read())
    
USER='santlond'
PASS=sp_config['ID']
DSN='IMPALA_PROD'
LOGDIR= 'logs'
# sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp")
sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp", spark_submit="spark3-submit")
 
# sparky = Sparky(username=USER, password=PASS, dsn=DSN)

helper = sparky.helper

/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2025-12-31 08:53:22 - [WARNING] - No se encontro la carpeta "/Users/santlond/Documents/ADQUIRENCIA_FERIA_EVA/logs" para guardar los logs


 ____  _____ __  __  ___ _____ _____ 
|  _ \| ____|  \/  |/ _ \_   _| ____|
| |_) |  _| | |\/| | | | || | |  _|  
|  _ <| |___| |  | | |_| || | | |___ 
|_| \_\_____|_|  |_|\___/ |_| |_____|
                                     
 ____  ____   _    ____  _  __
/ ___||  _ \ / \  |  _ \| |/ /
\___ \| |_) / _ \ | |_) | ' / 
 ___) |  __/ ___ \|  _ <| . \ 
|____/|_| /_/   \_\_| \_\_|\_\
                              



In [2]:
helper.obtener_ultima_ingestion('resultados_vspc_clientes.master_customer_data')

2025-12-31 08:54:39 - [INFO] - Buscando fechas para resultados_vspc_clientes.master_customer_data
2025-12-31 08:54:39 - [INFO] - Transcurrido: 1767189280, Tiempo de Refresco = 1000
/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:476: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, cn)
/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:476: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, cn)
/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:476: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or 

{'year': 2025, 'month': 12, 'day': 29}

# Introducción

Se evidencia que los registros en la tabla `resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf` las vinculaciones de adquirencia no comparte registros con la tabla `resultados_wompi.wompi_merchants` que contiene las vinculaciones a wompi

In [72]:
# HASTA QUE AÑO MES SE HAN ACTUALIZADO LAS TRXS
periodo_actual_trxs = '202511'
periodo_actual_trxs

'202511'

# Evolución vinculación aceptación comercios [Adquirencia + Wompi]

## Adquirencia

In [3]:
dict_ult_ing_adqu_vinc = helper.obtener_ultima_ingestion('resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf')
dict_ult_ing_adqu_vinc

2025-12-31 08:54:46 - [INFO] - Buscando fechas para resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf
/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:476: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, cn)
2025-12-31 08:54:46 - [INFO] - Finalizo la busqueda, duracion: 00:00.4, resultado: {'year': 2025, 'month': 12, 'day': 21}


{'year': 2025, 'month': 12, 'day': 21}

In [7]:
sql_drop = """DROP TABLE IF EXISTS proceso.mdo_aceptacion_comercios_hist_vinc_adqu PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_aceptacion_comercios_hist_vinc_adqu STORED AS PARQUET AS WITH vinculados AS
  (SELECT codigo_unico,
          min(periodo) AS fecha_ym
   FROM resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf
   WHERE YEAR <= """ + str(dict_ult_ing_adqu_vinc['year']) + """
     AND MONTH BETWEEN 1 AND 12
     AND DAY BETWEEN 1 AND 31
     AND periodo IS NOT NULL
   GROUP BY 1),
                                                                                       conteo AS
  (SELECT fecha_ym,
          count(*) AS num_vinc_new,
          cast(left(cast(fecha_ym AS STRING), 4) AS int) AS YEAR,
          cast(right(cast(fecha_ym AS STRING), 2) AS int) AS mes
   FROM vinculados
   GROUP BY 1)
SELECT fecha_ym,
       num_vinc_new,
       sum(num_vinc_new) OVER (PARTITION BY YEAR
                           ORDER BY YEAR,
                                    mes ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS num_vinc_new_cumsum_ym,
                          'adquirencia' AS producto
FROM conteo
ORDER BY fecha_ym DESC;
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_aceptacion_comercios_hist_vinc_adqu;"""
helper.ejecutar_consulta(sql_compute)

---------------------------------------------------------------------------------------------
  i   tipo                    nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 4/4    DROP ...o_aceptacion_comercios_hist_vinc_adqu   finalizado   11:46:02 AM     00:00.2 
---------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------
  i   tipo                    nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 5/5  CREATE ...o_aceptacion_comercios_hist_vinc_adqu   finalizado   11:46:02 AM     00:07.8 
---------------------------------------------------------------------------------------------
------------------------------------------------------------

## Wompi

In [4]:
dict_ult_ing_wompi_merch = helper.obtener_ultima_ingestion('resultados_wompi.wompi_merchants')
dict_ult_ing_wompi_merch


2025-12-19 11:38:09 - [INFO] - Buscando fechas para resultados_wompi.wompi_merchants
/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:476: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, cn)
2025-12-19 11:38:10 - [INFO] - Finalizo la busqueda, duracion: 00:00.9, resultado: {'year': 2025, 'month': 12, 'day': 19}


{'year': 2025, 'month': 12, 'day': 19}

In [54]:
sql_drop = """DROP TABLE IF EXISTS proceso.mdo_aceptacion_comercios_hist_vinc_wompi PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_aceptacion_comercios_hist_vinc_wompi STORED AS PARQUET AS
WITH conteo AS (
SELECT CASt(left(cast(creado as string), 6) as int) as fecha_ym, 
       count(*) as num_vinc_new
FROM resultados_wompi.wompi_merchants
WHERE YEAR = """ + str(dict_ult_ing_wompi_merch['year']) + """
  AND MONTH = """ + str(dict_ult_ing_wompi_merch['month']) + """
  AND DAY = """ + str(dict_ult_ing_wompi_merch['day']) + """
  AND modelo = 'Agregador'
  and activo = 'A'
  and desembolsos_permitidos = 'Si'
GROUP BY 1
), conteo_y_m AS (
SELECT fecha_ym,
        num_vinc_new,
        cast(left(cast(fecha_ym AS STRING), 4) AS int) AS YEAR,
        cast(right(cast(fecha_ym AS STRING), 2) AS int) AS mes
FROM conteo
)
SELECT fecha_ym,
       num_vinc_new,
       sum(num_vinc_new) OVER (PARTITION BY YEAR
                           ORDER BY YEAR,
                                    mes ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS num_vinc_new_cumsum_ym,
        'wompi' AS producto
FROM conteo_y_m
ORDER BY fecha_ym DESC;
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_aceptacion_comercios_hist_vinc_wompi;"""
helper.ejecutar_consulta(sql_compute)

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 77/77      DROP ..._aceptacion_comercios_hist_vinc_wompi   finalizado   11:43:08 PM     00:00.4 
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 78/78    CREATE ..._aceptacion_comercios_hist_vinc_wompi   finalizado   11:43:09 PM     00:00.7 
-------------------------------------------------------------------------------------------------
--------------------

# Vinculación aceptación comercios

In [55]:
sql_drop = """DROP TABLE IF EXISTS proceso.mdo_aceptacion_comercios_vinc PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_aceptacion_comercios_vinc STORED AS PARQUET AS
WITH junte AS
  (SELECT fecha_ym,
          num_vinc_new,
          num_vinc_new_cumsum_ym,
          producto
   FROM proceso.mdo_aceptacion_comercios_hist_vinc_adqu
   UNION ALL SELECT fecha_ym,
                    num_vinc_new,
                    num_vinc_new_cumsum_ym,
                    producto
   FROM proceso.mdo_aceptacion_comercios_hist_vinc_wompi),
     agregado AS
  (SELECT fecha_ym,
          cast(left(cast(fecha_ym AS STRING), 4) AS int) AS YEAR,
          cast(right(cast(fecha_ym AS STRING), 2) AS int) AS mes,
          sum(num_vinc_new) AS num_vinc_new,
          1 AS secuencia2
   FROM junte
   GROUP BY 1,
            2,
            3)
SELECT fecha_ym,
       num_vinc_new,
       sum(num_vinc_new) OVER (PARTITION BY secuencia2 ORDER BY fecha_ym ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS num_vinc_cumsum,
       sum(num_vinc_new) OVER (PARTITION BY YEAR
                           ORDER BY YEAR,
                                    mes ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS num_vinc_new_cumsum_ym
FROM agregado
ORDER BY fecha_ym DESC;
"""
df_outcome = helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_aceptacion_comercios_vinc;"""
helper.ejecutar_consulta(sql_compute)

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 80/80      DROP    proceso.mdo_aceptacion_comercios_vinc   finalizado   11:43:11 PM     00:00.3 
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 81/81    CREATE    proceso.mdo_aceptacion_comercios_vinc   finalizado   11:43:11 PM     00:00.5 
-------------------------------------------------------------------------------------------------
--------------------

In [56]:
sql = """
SELECT fecha_ym,
       num_vinc_new,
       num_vinc_cumsum,
       num_vinc_new_cumsum_ym
FROM proceso.mdo_aceptacion_comercios_vinc
ORDER BY fecha_ym DESC
"""
df_outcome = helper.obtener_dataframe(sql)

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 83/83 DATAFRAME                                           descargando   11:43:13 PM             

2025-12-11 23:43:14 - [INFO] - 87 filas, 4 columnas, 00:01.1 consultando, 00:00.2 descargando, 00:00.0 convirtiendo


 83/83 DATAFRAME                                            finalizado   11:43:13 PM     00:01.5 
-------------------------------------------------------------------------------------------------


In [57]:
df_outcome[40:].head(20)

,fecha_ym,num_vinc_new,num_vinc_cumsum,num_vinc_new_cumsum_ym
40,202208.0,6303,161179,40580
41,202207.0,5110,154876,34277
42,202206.0,5804,149766,29167
43,202205.0,5311,143962,23363
44,202204.0,4498,138651,18052
45,202203.0,5760,134153,13554
46,202202.0,4404,128393,7794
47,202201.0,3390,123989,3390
48,202112.0,4711,120599,68238
49,202111.0,7857,115888,63527


# Uso de vinculados nuevos aceptación comercios

In [58]:
sql_drop = """DROP TABLE IF EXISTS proceso.mdo_aceptacion_comercios_num_vinc_new_uso PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_aceptacion_comercios_num_vinc_new_uso STORED AS PARQUET AS
WITH uso_adqui AS
  (SELECT periodo,
          count(*) AS num_vinc_new_uso_adqu
   FROM proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist
   WHERE producto = 'adqui'
   AND tipo_cliente = 'nuevos'
   GROUP BY 1),
     uso_wompi AS
  (SELECT periodo,
          count(*) AS num_vinc_new_uso_womp
   FROM proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist
   WHERE producto = 'wompi'
   AND tipo_cliente = 'nuevos'
   GROUP BY 1)
SELECT a.periodo as fecha_ym,
       a.num_vinc_new_uso_adqu + nvl(b.num_vinc_new_uso_womp, 0) AS num_vinc_new_uso_cumsum_ym
FROM uso_adqui AS a
LEFT JOIN uso_wompi AS b ON a.periodo = b.periodo
ORDER BY a.periodo DESC;
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_aceptacion_comercios_num_vinc_new_uso;"""
helper.ejecutar_consulta(sql_compute)

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 84/84      DROP ...aceptacion_comercios_num_vinc_new_uso   finalizado   11:43:14 PM     00:00.2 
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 85/85    CREATE ...aceptacion_comercios_num_vinc_new_uso   finalizado   11:43:15 PM     00:01.5 
-------------------------------------------------------------------------------------------------
--------------------

# Uso de vinculados viejos aceptación comercios

In [59]:
sql_drop = """DROP TABLE IF EXISTS proceso.mdo_aceptacion_comercios_num_vinc_old_uso PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_aceptacion_comercios_num_vinc_old_uso STORED AS PARQUET AS
WITH uso_adqui AS
  (SELECT periodo,
          count(*) AS num_vinc_old_uso_adqu
   FROM proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist
   WHERE producto = 'adqui'
   AND tipo_cliente = 'viejos'
   GROUP BY 1),
     uso_wompi AS
  (SELECT periodo,
          count(*) AS num_vinc_old_uso_womp
   FROM proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist
   WHERE producto = 'wompi'
   AND tipo_cliente = 'viejos'
   GROUP BY 1)
SELECT a.periodo as fecha_ym,
       a.num_vinc_old_uso_adqu + nvl(b.num_vinc_old_uso_womp, 0) AS num_vinc_old_uso_cumsum_ym
FROM uso_adqui AS a
LEFT JOIN uso_wompi AS b ON a.periodo = b.periodo
ORDER BY a.periodo DESC;
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_aceptacion_comercios_num_vinc_old_uso;"""
helper.ejecutar_consulta(sql_compute)

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 87/87      DROP ...aceptacion_comercios_num_vinc_old_uso   finalizado   11:43:17 PM     00:00.3 
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 88/88    CREATE ...aceptacion_comercios_num_vinc_old_uso   finalizado   11:43:18 PM     00:02.1 
-------------------------------------------------------------------------------------------------
--------------------

# Uso de vinculados todos aceptación comercios

In [60]:
sql_drop = """DROP TABLE IF EXISTS proceso.mdo_aceptacion_comercios_num_vinc_all_uso PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_aceptacion_comercios_num_vinc_all_uso STORED AS PARQUET AS
WITH uso_adqui AS
  (SELECT periodo,
          count(*) AS num_vinc_all_uso_adqu
   FROM proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist
   WHERE producto = 'adqui'
   AND tipo_cliente = 'todos'
   GROUP BY 1),
     uso_wompi AS
  (SELECT periodo,
          count(*) AS num_vinc_all_uso_womp
   FROM proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist
   WHERE producto = 'wompi'
   AND tipo_cliente = 'todos'
   GROUP BY 1)
SELECT a.periodo as fecha_ym,
       a.num_vinc_all_uso_adqu + nvl(b.num_vinc_all_uso_womp, 0) AS num_vinc_all_uso_cumsum_ym
FROM uso_adqui AS a
LEFT JOIN uso_wompi AS b ON a.periodo = b.periodo
ORDER BY a.periodo DESC;
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_aceptacion_comercios_num_vinc_all_uso;"""
helper.ejecutar_consulta(sql_compute)

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 90/90      DROP ...aceptacion_comercios_num_vinc_all_uso   finalizado   11:43:21 PM     00:00.3 
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 91/91    CREATE ...aceptacion_comercios_num_vinc_all_uso   finalizado   11:43:21 PM     00:02.6 
-------------------------------------------------------------------------------------------------
--------------------

# Tabla resultado

In [80]:
sql = """
WITH outcome1 AS
  (SELECT a.fecha_ym,
          a.num_vinc_new,
          a.num_vinc_cumsum,
          a.num_vinc_new_cumsum_ym,
          nvl(b.num_vinc_new_uso_cumsum_ym, 0) AS num_vinc_new_uso_cumsum_ym,
          round(nvl(b.num_vinc_new_uso_cumsum_ym, 0)/a.num_vinc_new_cumsum_ym, 4) AS num_vinc_new_prop_uso,
          nvl(c.num_vinc_old_uso_cumsum_ym, 0) AS num_vinc_old_uso_cumsum_ym,
          nvl(d.num_vinc_all_uso_cumsum_ym, 0) AS num_vinc_all_uso_cumsum_ym,
          round(nvl(d.num_vinc_all_uso_cumsum_ym, 0)/a.num_vinc_cumsum, 4) AS num_vinc_all_prop_uso,
          left(cast(a.fecha_ym AS string), 4) AS YEAR,
          right(cast(a.fecha_ym AS string), 2) AS mes
   FROM proceso.mdo_aceptacion_comercios_vinc AS a
   LEFT JOIN proceso.mdo_aceptacion_comercios_num_vinc_new_uso AS b ON a.fecha_ym = b.fecha_ym
   LEFT JOIN proceso.mdo_aceptacion_comercios_num_vinc_old_uso AS c ON a.fecha_ym = c.fecha_ym
   LEFT JOIN proceso.mdo_aceptacion_comercios_num_vinc_all_uso AS d ON a.fecha_ym = d.fecha_ym),
     outcome2 AS
  (SELECT fecha_ym,
          num_vinc_cumsum AS num_vinc_old,
          cast(cast(YEAR AS int) + 1 AS string) AS YEAR
   FROM outcome1
   WHERE mes = '12')
SELECT a.fecha_ym,
       CONCAT(a.YEAR, '/', a.mes, '/', '01') AS fecha_ym2,
       a.num_vinc_new,
       a.num_vinc_new_cumsum_ym,
       a.num_vinc_new_uso_cumsum_ym,
       a.num_vinc_new_prop_uso,
       b.num_vinc_old,
       a.num_vinc_old_uso_cumsum_ym,
       round(a.num_vinc_old_uso_cumsum_ym/b.num_vinc_old, 4) AS num_vinc_old_prop_uso,
       a.num_vinc_cumsum,
       a.num_vinc_all_uso_cumsum_ym,
       a.num_vinc_all_prop_uso
FROM outcome1 AS a
LEFT JOIN outcome2 AS b ON a.year = b.year
WHERE a.fecha_ym BETWEEN 202201 AND 202511
ORDER BY a.fecha_ym DESC;
"""
# print(sql)
df_outcome = helper.obtener_dataframe(sql)
df_outcome

---------------------------------------------------------------------------------------------------
    i      tipo                     nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------------
 100/100 DATAFRAME                                           descargando   12:14:47 AM             

2025-12-12 00:14:48 - [INFO] - 47 filas, 12 columnas, 00:00.6 consultando, 00:00.2 descargando, 00:00.0 convirtiendo


 100/100 DATAFRAME                                            finalizado   12:14:47 AM     00:00.9 
---------------------------------------------------------------------------------------------------


,fecha_ym,fecha_ym2,num_vinc_new,num_vinc_new_cumsum_ym,num_vinc_new_uso_cumsum_ym,num_vinc_new_prop_uso,num_vinc_old,num_vinc_old_uso_cumsum_ym,num_vinc_old_prop_uso,num_vinc_cumsum,num_vinc_all_uso_cumsum_ym,num_vinc_all_prop_uso
0,202511.0,2025/11/01,14983,98681,37253,0.3775,309667,96086,0.3103,408348,133339,0.3265
1,202510.0,2025/10/01,16180,83698,32665,0.3903,309667,95557,0.3086,393365,128222,0.3260
2,202509.0,2025/09/01,18355,67518,27900,0.4132,309667,94927,0.3065,377185,122827,0.3256
3,202508.0,2025/08/01,10621,49163,22995,0.4677,309667,94168,0.3041,358830,117163,0.3265
4,202507.0,2025/07/01,7876,38542,18936,0.4913,309667,93317,0.3013,348209,112253,0.3224
5,202506.0,2025/06/01,6191,30666,15268,0.4979,309667,92331,0.2982,340333,107599,0.3162
6,202505.0,2025/05/01,6337,24475,12327,0.5037,309667,91225,0.2946,334142,103552,0.3099
7,202504.0,2025/04/01,5104,18138,9031,0.4979,309667,89479,0.2890,327805,98510,0.3005
8,202503.0,2025/03/01,4731,13034,6196,0.4754,309667,87526,0.2826,322701,93722,0.2904
9,202502.0,2025/02/01,4296,8303,3760,0.4528,309667,84511,0.2729,317970,88271,0.2776


In [81]:
df_outcome.to_excel('main_data/evolucion_vinculacion_y_uso_adquirencia_y_wompi.xlsx')